# Corridas de la funcion de las seis jorobas de camello

Este notebook ejecuta corridas de descenso por gradiente para la funcion de las seis jorobas de camello en `2D`, usando `n = 100, 500 y 1000`.

Los resultados se guardan automaticamente en la carpeta `datos/`.

In [ ]:
import csv
import json
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import matplotlib.pyplot as plt
import numpy as np

from funciones_gradientes import get_initial_position, run_gradient_descent, six_hump_camel_gradient
from funciones_objetivo import six_hump_camel

In [ ]:
N_CORRIDAS_LISTA = [100, 500, 1000]
DIMENSION = 2
LIMITS = (-3.0, 3.0)
RATE = 1e-3
MAX_ITERATIONS = 1000
TOLERANCE = 1e-6
SEED = 42

if DIMENSION != 2:
    raise ValueError("La funcion de las seis jorobas de camello solo esta definida aqui para 2D.")

if any(n <= 0 for n in N_CORRIDAS_LISTA):
    raise ValueError("Todos los valores de N_CORRIDAS_LISTA deben ser positivos.")

In [ ]:
def contar_evaluaciones(resultado: dict) -> int:
    """Cuenta evaluaciones de la funcion objetivo hechas durante la corrida."""
    return len(resultado["function_values"]) + 1


def ejecutar_corridas(n_corridas: int) -> list[dict]:
    """Ejecuta las corridas de la funcion de las seis jorobas de camello en 2D."""
    resultados = []

    for corrida in range(1, n_corridas + 1):
        posicion_inicial = get_initial_position(DIMENSION, limits=LIMITS)
        resultado = run_gradient_descent(
            initial_position=posicion_inicial,
            function=six_hump_camel,
            gradient_function=six_hump_camel_gradient,
            rate=RATE,
            max_iterations=MAX_ITERATIONS,
            tolerance=TOLERANCE,
        )

        resultados.append(
            {
                "corrida": corrida,
                "dimension": DIMENSION,
                "posicion_inicial": resultado["initial_position"].tolist(),
                "posicion_final": resultado["final_position"].tolist(),
                "valor_final": float(resultado["final_value"]),
                "iteraciones": int(resultado["iterations"]),
                "evaluaciones": contar_evaluaciones(resultado),
            }
        )

    return resultados


def guardar_resultados(resultados: list[dict], n_corridas: int) -> tuple[Path, Path]:
    """Guarda los resultados en JSON y CSV usando nombres dinamicos."""
    directorio_salida = Path("datos")
    directorio_salida.mkdir(exist_ok=True)

    base_name = f"six_hump_camel_2d_n{n_corridas}"
    json_path = directorio_salida / f"{base_name}.json"
    csv_path = directorio_salida / f"{base_name}.csv"

    with json_path.open("w", encoding="utf-8") as json_file:
        json.dump(resultados, json_file, indent=2)

    with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(
            csv_file,
            fieldnames=[
                "corrida",
                "dimension",
                "posicion_inicial",
                "posicion_final",
                "valor_final",
                "iteraciones",
                "evaluaciones",
            ],
        )
        writer.writeheader()
        writer.writerows(resultados)

    return json_path, csv_path

In [ ]:
np.random.seed(SEED)

resultados_por_n = {}

for n_corridas in N_CORRIDAS_LISTA:
    resultados = ejecutar_corridas(n_corridas)
    json_path, csv_path = guardar_resultados(resultados, n_corridas)
    resultados_por_n[n_corridas] = {
        "resultados": resultados,
        "json_path": json_path,
        "csv_path": csv_path,
    }

    print(f"Listo: seis jorobas de camello 2D con n={n_corridas}")
    print(f"  JSON: {json_path}")
    print(f"  CSV:  {csv_path}")


In [ ]:
for n_corridas in N_CORRIDAS_LISTA:
    resultados = resultados_por_n[n_corridas]["resultados"]
    valores_finales = np.array([fila["valor_final"] for fila in resultados], dtype=float)
    evaluaciones = np.array([fila["evaluaciones"] for fila in resultados], dtype=int)
    iteraciones = np.array([fila["iteraciones"] for fila in resultados], dtype=int)

    print(f"Resumen seis jorobas de camello 2D - n={n_corridas}")
    print(f"Mejor valor final: {valores_finales.min():.8f}")
    print(f"Peor valor final: {valores_finales.max():.8f}")
    print(f"Promedio valor final: {valores_finales.mean():.8f}")
    print(f"Promedio evaluaciones: {evaluaciones.mean():.2f}")
    print(f"Promedio iteraciones: {iteraciones.mean():.2f}")
    print("-" * 50)

In [ ]:
fig, axes = plt.subplots(2, len(N_CORRIDAS_LISTA), figsize=(5 * len(N_CORRIDAS_LISTA), 8))

for idx, n_corridas in enumerate(N_CORRIDAS_LISTA):
    resultados = resultados_por_n[n_corridas]["resultados"]
    valores_finales = np.array([fila["valor_final"] for fila in resultados], dtype=float)
    evaluaciones = np.array([fila["evaluaciones"] for fila in resultados], dtype=int)

    axes[0, idx].hist(valores_finales, bins=20, color="steelblue", edgecolor="black")
    axes[0, idx].set_title(f"Valor final - n={n_corridas}")
    axes[0, idx].set_xlabel("Valor final")
    axes[0, idx].set_ylabel("Frecuencia")

    axes[1, idx].hist(evaluaciones, bins=20, color="indianred", edgecolor="black")
    axes[1, idx].set_title(f"Evaluaciones - n={n_corridas}")
    axes[1, idx].set_xlabel("Evaluaciones")
    axes[1, idx].set_ylabel("Frecuencia")

fig.suptitle("Seis jorobas de camello 2D - comparacion de corridas")
fig.tight_layout()
plt.show()

In [ ]:
n_corridas = N_CORRIDAS_LISTA[-1]
resultados = resultados_por_n[n_corridas]["resultados"]
valores_finales = np.array([fila["valor_final"] for fila in resultados], dtype=float)
iteraciones = np.array([fila["iteraciones"] for fila in resultados], dtype=int)
posiciones_finales = np.array([fila["posicion_final"] for fila in resultados], dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

scatter = axes[0].scatter(
    posiciones_finales[:, 0],
    posiciones_finales[:, 1],
    c=valores_finales,
    cmap="viridis",
    edgecolors="black",
)
axes[0].set_title("Posiciones finales en 2D")
axes[0].set_xlabel("x1 final")
axes[0].set_ylabel("x2 final")
fig.colorbar(scatter, ax=axes[0], shrink=0.8, label="Valor final")

axes[1].boxplot([valores_finales, iteraciones], tick_labels=["Valor final", "Iteraciones"])
axes[1].set_title("Resumen de dispersion - 2D")

plt.suptitle(f"Seis jorobas de camello 2D - posiciones finales (n={n_corridas})")
plt.tight_layout()
plt.show()